# 06 — Stage 4: Safety Repair via ReSta

**Goal.** Repair models flagged by the Stage 3 hardened check by adding a scaled safety task vector $\gamma (\theta_{\text{safe}} - \theta_{\text{base}})$ derived from `CohereLabs/tiny-aya-global` and `CohereLabs/tiny-aya-base`, restoring both refusal behavior and genuine refusal fingerprint alignment.

In [ ]:
!git clone https://github.com/mathcrypto/forged-fingerprints.git /content/forged-fingerprints 2>/dev/null || (cd /content/forged-fingerprints && git pull)
%cd /content/forged-fingerprints
!pip uninstall -y torchao -q
!pip install -q -e . peft accelerate

In [ ]:
import os
from huggingface_hub import login, snapshot_download

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')
login(token=token) if token else login()

HF_ORG = 'team-doom'
snapshot_download(f'{HF_ORG}/forged-fingerprints-contrast-public', repo_type='dataset', local_dir='data/contrast_sets/public')
snapshot_download(f'{HF_ORG}/forged-fingerprints-results', repo_type='dataset', local_dir='outputs')

## 1. Run ReSta Safety Vector Repair on Flagged Models

In [ ]:
from pathlib import Path
import subprocess

targets = [
    Path('outputs/zoo/harmful_100/adapter'),
    Path('outputs/attacks/stage2a_last_layers/adapter'),
    Path('outputs/attacks/stage2b_lambda_sweep_lam2.0/adapter'),
]

for adapter_dir in targets:
    if adapter_dir.exists():
        name = adapter_dir.parent.name
        out_file = f'outputs/repair/{name}_repaired.json'
        print(f'=== Repairing {name} ===')
        subprocess.run([
            'python', 'scripts/run_repair.py',
            '--adapter', str(adapter_dir),
            '--gamma', '0.5',
            '--out', out_file
        ], check=True)

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi

print(f"{'repaired_model':32s} {'before_refusal':>14s} {'after_refusal':>14s} {'after_cosine':>13s}")
print('-' * 78)
for p in sorted(Path('outputs/repair').glob('*.json')):
    r = json.loads(p.read_text())
    print(f"{p.stem:32s} {r['before_refusal_rate']:14.3f} {r['after_refusal_rate']:14.3f} {r['after_cosine_to_ref']:13.4f}")

HfApi().upload_folder(folder_path='outputs/repair', path_in_repo='repair', repo_id=f'{HF_ORG}/forged-fingerprints-results', repo_type='dataset')